# Exercise 2.11 — Vacuum Cleaner World (PEAS Simulator)

This notebook implements a **performance-measuring environment simulator** for the classic **vacuum-cleaner world**.

**Goal (Exercise 2.11):**
- Build the *environment simulator* (state, transition function, percepts, performance measure)
- Keep it **modular** so sensors/actions/environment can be changed easily
- Run at least **two agents** (e.g., reflex vs random) and compare performance over multiple trials

---
## PEAS Specification (explicit)

**P — Performance Measure**
- +1 per time step **for each clean square** (reward for cleanliness)
- −1 per **non-NOOP** action (small action cost)
- +5 bonus when **SUCK** successfully cleans dirt (cleaning event bonus)

**E — Environment**
- Discrete set of locations (minimum: `A`, `B`)
- Dirt distribution per location
- Agent location
- Transition function updates state based on action (environment controls state changes)

**A — Actuators**
- `LEFT`, `RIGHT`, `SUCK`, `NOOP`

**S — Sensors**
- Percept = `(location, dirty_here)`
- Sensor model is modular so you can swap in a noisy/partial sensor later


## Import used to:

- create a RandomAgent
- randomize initial states across multiple trials (required experiment)

In [ ]:
import random

## 1) Performance Measure (P):
- Tracks a cumulative score over time.
- This is separate from the agent and environment to satisfy the architecture requirement.
- update(...) is called each time step after the environment applies the action.

In [ ]:
class PerformanceEvaluator:
    """Tracks and updates cumulative performance score."""

    def __init__(self, clean_reward_per_square=1, action_cost=1, clean_event_bonus=5):
        self.clean_reward_per_square = clean_reward_per_square
        self.action_cost = action_cost
        self.clean_event_bonus = clean_event_bonus
        self.total = 0

    def update(self, state_after_action, action, cleaned_event):
        # +1 per clean square each time step
        clean_squares = sum(1 for loc, dirty in state_after_action.dirt.items() if not dirty)
        self.total += self.clean_reward_per_square * clean_squares

        # -1 per action (except NOOP)
        if action != Action.NOOP:
            self.total -= self.action_cost

        # +bonus if SUCK cleaned something
        if cleaned_event:
            self.total += self.clean_event_bonus

## 2.1) Environment State (E)

- Stores everything the environment needs to know at time t:
  - agent location
  - dirt distribution per location
  - current time step
- This is NOT modified directly by the agent; the environment updates it.

In [ ]:
class VacuumState:
    """Stores the environment state (agent location + dirt map + time step)."""

    def __init__(self, agent_loc, dirt_map, step=0):
        # agent_loc: str, e.g., "A" or "B"
        # dirt_map: dict {location: bool} where True means DIRTY
        self.agent_loc = agent_loc
        self.dirt = dict(dirt_map)
        self.step = step

    def copy(self):
        return VacuumState(self.agent_loc, dict(self.dirt), self.step)

    def __repr__(self):
        return f"VacuumState(step={self.step}, agent_loc={self.agent_loc}, dirt={self.dirt})"

## 2.2) Environment (E):
- Controls state transitions and percept generation.
- Key requirement: the agent does NOT modify the environment state directly.
- The transition function is defined here and is independent of the agent implementation.

In [ ]:
class VacuumEnvironment:
    """Environment controls state transitions and percept generation.

    Architectural rule:
    - The agent does NOT modify environment state directly.
    - The environment applies the action via transition.
    """

    def __init__(self, locations, adjacency, sensor_model=None):
        self.locations = list(locations)
        self.adjacency = dict(adjacency)
        self.sensor = sensor_model if sensor_model is not None else SensorModel()

    def initial_state(self, agent_loc, dirt_map):
        # Ensure every location exists in the dirt map
        full_dirt = {loc: bool(dirt_map.get(loc, False)) for loc in self.locations}
        return VacuumState(agent_loc=agent_loc, dirt_map=full_dirt, step=0)

    def get_percept(self, state):
        return self.sensor.percept(state)

    def transition(self, state, action):
        """Apply action to state and return (new_state, cleaned_event)."""
        new_state = state.copy()
        new_state.step = state.step + 1
        cleaned_event = False

        # Movement: if action is a move and the adjacency map defines it, move; otherwise stay.
        moves = self.adjacency.get(state.agent_loc, {})
        if action in moves:
            new_state.agent_loc = moves[action]

        elif action == Action.SUCK:
            loc = state.agent_loc
            if new_state.dirt.get(loc, False):
                new_state.dirt[loc] = False
                cleaned_event = True

        elif action == Action.NOOP:
            pass

        return new_state, cleaned_event

## 3) Actuators (A):

- Defines the actions available to any agent.
- Keeping it in one place makes it modular (we can add new actions later).

In [ ]:
class Action:
    # Movement actions
    LEFT = "LEFT"
    RIGHT = "RIGHT"
    UP = "UP"
    DOWN = "DOWN"

    # Cleaning / control actions
    SUCK = "SUCK"
    NOOP = "NOOP"

    MOVES = [LEFT, RIGHT, UP, DOWN]
    ALL = MOVES + [SUCK, NOOP]


## 4) Sensors (S):
- A sensor model converts the true environment state into a percept given to the agent.
- Modular design: we can swap SensorModel() for NoisySensorModel() without changing agents.
- Percept format: (location, dirty_here)

In [ ]:
class SensorModel:
    """Base sensor model: returns (location, dirty_here)."""

    def percept(self, state):
        loc = state.agent_loc
        dirty_here = bool(state.dirt.get(loc, False))
        return (loc, dirty_here)


class NoisySensorModel(SensorModel):
    """Optional sensor model: flips dirty/clean with probability flip_prob."""

    def __init__(self, flip_prob=0.1, seed=None):
        self.flip_prob = flip_prob
        self.rng = random.Random(seed)

    def percept(self, state):
        loc, dirty_here = super().percept(state)
        if self.rng.random() < self.flip_prob:
            dirty_here = not dirty_here
        return (loc, dirty_here)

## 5) Agents:
- Agent programs map percepts -> actions.
- We implement two required agents:
  - SimpleReflexAgent (deterministic reflex policy)
  - RandomAgent (baseline randomized policy)
- Agents never access or modify the environment state directly.

In [ ]:
class Agent:
    def act(self, percept):
        raise NotImplementedError


class SimpleReflexAgent(Agent):
    """Policy:
    - If current square is dirty → SUCK
    - Else move to the other square (A↔B) deterministically
    """

    def act(self, percept):
        loc, dirty_here = percept
        if dirty_here:
            return Action.SUCK
        return Action.RIGHT if loc == "A" else Action.LEFT


class RandomAgent(Agent):
    """Chooses actions uniformly at random from a provided action set.

    Default action set matches the original 2-location vacuum world:
    {LEFT, RIGHT, SUCK, NOOP}
    """

    def __init__(self, seed=None, action_set=None):
        self.rng = random.Random(seed)
        self.action_set = action_set if action_set is not None else [Action.LEFT, Action.RIGHT, Action.SUCK, Action.NOOP]

    def act(self, percept):
        return self.rng.choice(self.action_set)


## 6) Simulator loop:
- Runs the interaction cycle: percept -> agent action -> environment transition -> performance update.
- This function enforces the required separation between: Agent program, Environment dynamics, and Performance evaluator.

In [ ]:
def run_simulation(env, agent, evaluator, init_state, max_steps=30, verbose=False):
    """Runs the environment-agent interaction loop."""
    state = init_state
    history = []  # (step, percept, action, score_after)

    for _ in range(max_steps):
        percept = env.get_percept(state)
        action = agent.act(percept)

        new_state, cleaned_event = env.transition(state, action)
        evaluator.update(new_state, action, cleaned_event)

        if verbose:
            print(f"t={state.step:02d} percept={percept} action={action:>4} -> score={evaluator.total}")

        history.append((state.step, percept, action, evaluator.total))
        state = new_state

    return evaluator.total, state, history

## 7) Build world and test
- Environment instantiation (2-location world):
  - Creates the minimum vacuum world A <-> B, sets an initial dirt configuration,
- Runs a first test with the SimpleReflexAgent using verbose output.

In [ ]:
# Two-location world A <-> B
locations = ["A", "B"]
adjacency = {
    "A": {Action.RIGHT: "B", Action.LEFT: "A"},
    "B": {Action.LEFT: "A", Action.RIGHT: "B"},
}

env = VacuumEnvironment(locations, adjacency, sensor_model=SensorModel())

# Example: both dirty, agent starts at A
init = env.initial_state(agent_loc="A", dirt_map={"A": True, "B": True})

# Reflex agent test
reflex_agent = SimpleReflexAgent()
eval_reflex = PerformanceEvaluator()
score_reflex, final_state_reflex, _ = run_simulation(env, reflex_agent, eval_reflex, init, max_steps=20, verbose=True)

print("\nReflex final score:", score_reflex)
print("Reflex final state:", final_state_reflex)

t=00 percept=('A', True) action=SUCK -> score=5
t=01 percept=('A', False) action=RIGHT -> score=5
t=02 percept=('B', True) action=SUCK -> score=11
t=03 percept=('B', False) action=LEFT -> score=12
t=04 percept=('A', False) action=RIGHT -> score=13
t=05 percept=('B', False) action=LEFT -> score=14
t=06 percept=('A', False) action=RIGHT -> score=15
t=07 percept=('B', False) action=LEFT -> score=16
t=08 percept=('A', False) action=RIGHT -> score=17
t=09 percept=('B', False) action=LEFT -> score=18
t=10 percept=('A', False) action=RIGHT -> score=19
t=11 percept=('B', False) action=LEFT -> score=20
t=12 percept=('A', False) action=RIGHT -> score=21
t=13 percept=('B', False) action=LEFT -> score=22
t=14 percept=('A', False) action=RIGHT -> score=23
t=15 percept=('B', False) action=LEFT -> score=24
t=16 percept=('A', False) action=RIGHT -> score=25
t=17 percept=('B', False) action=LEFT -> score=26
t=18 percept=('A', False) action=RIGHT -> score=27
t=19 percept=('B', False) action=LEFT -> scor

- Second required agent test: Runs the RandomAgent on the same initial environment to compare behavior and score.

In [ ]:
# Random agent test
rand_agent = RandomAgent(seed=42)
eval_rand = PerformanceEvaluator()
score_rand, final_state_rand, _ = run_simulation(env, rand_agent, eval_rand, init, max_steps=20, verbose=True)

print("\nRandom final score:", score_rand)
print("Random final state:", final_state_rand)

t=00 percept=('A', True) action=LEFT -> score=-1
t=01 percept=('A', True) action=LEFT -> score=-2
t=02 percept=('A', True) action=SUCK -> score=3
t=03 percept=('A', False) action=RIGHT -> score=3
t=04 percept=('B', True) action=RIGHT -> score=3
t=05 percept=('B', True) action=RIGHT -> score=3
t=06 percept=('B', True) action=LEFT -> score=3
t=07 percept=('A', False) action=LEFT -> score=3
t=08 percept=('A', False) action=NOOP -> score=4
t=09 percept=('A', False) action=LEFT -> score=4
t=10 percept=('A', False) action=LEFT -> score=4
t=11 percept=('A', False) action=LEFT -> score=4
t=12 percept=('A', False) action=RIGHT -> score=4
t=13 percept=('B', True) action=RIGHT -> score=4
t=14 percept=('B', True) action=LEFT -> score=4
t=15 percept=('A', False) action=RIGHT -> score=4
t=16 percept=('B', True) action=NOOP -> score=5
t=17 percept=('B', True) action=RIGHT -> score=5
t=18 percept=('B', True) action=NOOP -> score=6
t=19 percept=('B', True) action=SUCK -> score=12

Random final score: 1

## 8) Multiple trials experiment:
- Runs many randomized initial environments to compute average performance.
- This is required to compare agents fairly across different initial conditions.

In [ ]:
def average_score_over_trials(env, agent_factory, trials=30, max_steps=30, seed=0):
    rng = random.Random(seed)
    scores = []

    for _ in range(trials):
        # Randomize initial state each trial
        agent_loc = rng.choice(env.locations)
        dirt_map = {loc: rng.choice([True, False]) for loc in env.locations}
        init_state = env.initial_state(agent_loc, dirt_map)

        evaluator = PerformanceEvaluator()
        agent = agent_factory()
        score, _, _ = run_simulation(env, agent, evaluator, init_state, max_steps=max_steps, verbose=False)
        scores.append(score)

    return sum(scores) / len(scores), scores

avg_reflex, reflex_scores = average_score_over_trials(env, lambda: SimpleReflexAgent(), trials=50, max_steps=30, seed=1)
avg_random, random_scores = average_score_over_trials(env, lambda: RandomAgent(seed=123), trials=50, max_steps=30, seed=1)

print("Average score (Reflex):", avg_reflex)
print("Average score (Random):", avg_random)

Average score (Reflex): 34.18
Average score (Random): 32.84


---
## 10) How this notebook stays modular (what you will reuse in Exercise 2.14)

- Swap sensors: `SensorModel()` → `NoisySensorModel(flip_prob=0.2)`
- Change environment: edit `locations` and `adjacency` (later you can build a 2D grid adjacency)
- Add agents: create new classes inheriting `Agent`

In Exercise 2.14 you will reuse:
- `VacuumEnvironment`
- `PerformanceEvaluator`
- `run_simulation`
and you will focus on designing/testing more agent types and adversarial environments.


# Exercise 2.14 — Unknown Geography Vacuum World (2D + Obstacles)

This exercise extends the **same PEAS-based simulator** from Exercise 2.11 to a **2D grid** where:

- The geography (extent, boundaries, obstacles) is *unknown to the agent*.
- Dirt configuration is unknown.
- The agent can act: **Left, Right, Up, Down, Suck, NoOp**.
- The environment remains *fully specified internally* by its transition function; the agent learns only via percepts.

We will implement and compare:

1. **BerkeleyRandomVacuumAgent**: Completely random.
2. **BerkeleyReflexVacuumAgent**: Suck if dirty else move randomly.
3. **BerkeleyModelBasedVacuumAgent**: Maintains internal state to explore systematically and avoid known obstacles.


In [ ]:
from typing import Dict, Tuple, List, Set, Optional
Coord = Tuple[int, int]

MOVE_DELTAS: Dict[str, Tuple[int, int]] = {
    Action.LEFT: (-1, 0),
    Action.RIGHT: (1, 0),
    Action.UP: (0, -1),
    Action.DOWN: (0, 1),
}

def build_grid_adjacency(width: int, height: int, obstacles: Set[Coord]) -> Tuple[List[Coord], Dict[Coord, Dict[str, Coord]]]:
    """Build locations + adjacency for a 2D grid with blocked cells (obstacles)."""
    locations: List[Coord] = []
    adjacency: Dict[Coord, Dict[str, Coord]] = {}
    for y in range(height):
        for x in range(width):
            c = (x, y)
            if c in obstacles:
                continue
            locations.append(c)

    loc_set = set(locations)
    for (x, y) in locations:
        moves: Dict[str, Coord] = {}
        for act, (dx, dy) in MOVE_DELTAS.items():
            nx, ny = x + dx, y + dy
            nc = (nx, ny)
            if nc in loc_set:
                moves[act] = nc
            # If nc is outside bounds or obstacle -> undefined, env will keep agent in place.
        adjacency[(x, y)] = moves
    return locations, adjacency

def make_random_grid_environment(
    rng: random.Random,
    width_range=(4, 7),
    height_range=(4, 7),
    obstacle_prob=0.15,
    dirt_prob=0.5,
    sensor_model: Optional[SensorModel]=None
):
    """Create one random 2D vacuum world instance (unknown to the agent, known to the simulator)."""
    width = rng.randint(width_range[0], width_range[1])
    height = rng.randint(height_range[0], height_range[1])

    obstacles: Set[Coord] = set()
    for y in range(height):
        for x in range(width):
            if rng.random() < obstacle_prob:
                obstacles.add((x, y))

    # Ensure at least one free cell
    if len(obstacles) >= width * height:
        obstacles.discard((0, 0))

    locations, adjacency = build_grid_adjacency(width, height, obstacles)
    env = VacuumEnvironment(locations=locations, adjacency=adjacency, sensor_model=sensor_model)

    # Randomize agent start + dirt map
    agent_loc = rng.choice(locations)
    dirt_map = {loc: (rng.random() < dirt_prob) for loc in locations}
    init_state = env.initial_state(agent_loc, dirt_map)
    return env, init_state, {"width": width, "height": height, "obstacles": obstacles}


# Agent programs from UC Berkeley
# These agents mirror the UC Berkeley `agents.py` vacuum agents:
# - RandomVacuumAgent
# - ReflexVacuumAgent
# - ModelBasedVacuumAgent

class BerkeleyRandomVacuumAgent(Agent):
    # Chooses randomly from the available actuator actions. 

    def __init__(self, seed=None):
        self.rng = random.Random(seed)
        self.actions = [Action.LEFT, Action.RIGHT, Action.UP, Action.DOWN, Action.SUCK, Action.NOOP]

    def act(self, percept):
        return self.rng.choice(self.actions)


class BerkeleyReflexVacuumAgent(Agent):
    # Simple reflex: if dirty -> SUCK, else move (random walk).

    def __init__(self, seed=None):
        self.rng = random.Random(seed)
        self.moves = [Action.LEFT, Action.RIGHT, Action.UP, Action.DOWN]

    def act(self, percept):
        loc, dirty = percept
        if dirty:
            return Action.SUCK
        return self.rng.choice(self.moves)


class BerkeleyModelBasedVacuumAgent(Agent):
    # Model-based reflex agent (stateful exploration with backtracking).
    # Key idea (from Berkeley `ModelBasedVacuumAgent`): maintain an internal model
    # and choose NoOp if everything we've ever seen is clean AND we've explored all reachable cells we know.
    # The agent performs a DFS-like exploration, backtracking when it hits dead ends, and only stops when it has no reason to believe there could be more dirt.

    def __init__(self):
        # location -> dirty_bool (as last perceived)
        self.known_status: Dict[Coord, bool] = {}

        # learned transition model: (loc, action) -> next_loc  OR None if blocked
        self.T: Dict[Tuple[Coord, str], Optional[Coord]] = {}

        # Memory for exploration
        self.visited: Set[Coord] = set()
        self.parent: Dict[Coord, Coord] = {}              # DFS tree parent
        self.edge_to_parent: Dict[Coord, str] = {}
        self.last_loc: Optional[Coord] = None
        self.last_action: Optional[str] = None

        # deterministic action preference (keeps runs reproducible)
        self.move_order = [Action.UP, Action.RIGHT, Action.DOWN, Action.LEFT]

    def _update_model_from_transition(self, new_loc: Coord):
        """After receiving the new percept, update the learned transition model
        based on what happened after the last (move) action."""
        if self.last_loc is None or self.last_action is None:
            return
        if self.last_action not in MOVE_DELTAS:
            return

        key = (self.last_loc, self.last_action)
        
        if new_loc == self.last_loc:
            self.T[key] = None  # blocked / bump
        else:
            self.T[key] = new_loc
            if new_loc not in self.parent and new_loc != self.last_loc:
                self.parent[new_loc] = self.last_loc
                self.edge_to_parent[new_loc] = self.last_action

    def _action_to_reach(self, frm: Coord, to: Coord) -> Optional[str]:
        """Given learned transitions, find an action from frm that leads to to."""
        for act in self.move_order:
            if self.T.get((frm, act), None) == to:
                return act
        return None

    def act(self, percept):
        loc, dirty = percept

        # Update transition model from last step
        if self.last_loc is not None:
            self._update_model_from_transition(loc)

        # Update internal status model
        self.known_status[loc] = bool(dirty)
        self.visited.add(loc)

        # If dirty -> clean it now
        if dirty:
            self.last_loc = loc
            self.last_action = Action.SUCK
            return Action.SUCK
        
        if self.known_status and all(v is False for v in self.known_status.values()):
            # We can still keep exploring unknown space, but we won't stop until the agent has no reason to believe there could be more dirt.
            # We'll stop if there is no unexplored move from current location.
            has_untried = any((loc, act) not in self.T for act in self.move_order)
            if not has_untried:
                self.last_loc = loc
                self.last_action = Action.NOOP
                return Action.NOOP

        # Explore: try any untried movement from current location
        for act in self.move_order:
            if (loc, act) not in self.T:
                self.last_loc = loc
                self.last_action = act
                return act

        # No untried moves: try any known non-blocked move to an unvisited location
        for act in self.move_order:
            nxt = self.T.get((loc, act), None)
            if nxt is not None and nxt not in self.visited:
                self.last_loc = loc
                self.last_action = act
                return act

        # Otherwise backtrack along DFS tree if possible
        if loc in self.parent:
            p = self.parent[loc]
            act = self._action_to_reach(loc, p)
            if act is not None:
                self.last_loc = loc
                self.last_action = act
                return act

        # Fallback: any known non-blocked move
        for act in self.move_order:
            nxt = self.T.get((loc, act), None)
            if nxt is not None:
                self.last_loc = loc
                self.last_action = act
                return act

        self.last_loc = loc
        self.last_action = Action.NOOP
        return Action.NOOP


In [ ]:

import pandas as pd

def run_ex214_trials(
    trials=60,
    max_steps=150,
    seed=7,
    obstacle_prob=0.18,
    dirt_prob=0.55,
):
    rng = random.Random(seed)

    agents = {
        "Berkeley Random": lambda t: BerkeleyRandomVacuumAgent(seed=seed + 1000 + t),
        "Berkeley Reflex": lambda t: BerkeleyReflexVacuumAgent(seed=seed + 2000 + t),
        "Berkeley Model-Based": lambda t: BerkeleyModelBasedVacuumAgent(),
    }

    rows = []
    for t in range(trials):
        env, init_state, meta = make_random_grid_environment(
            rng=rng,
            obstacle_prob=obstacle_prob,
            dirt_prob=dirt_prob
        )

        for name, factory in agents.items():
            agent = factory(t)
            evaluator = PerformanceEvaluator(
                clean_reward_per_square=1,
                action_cost=1,
                clean_event_bonus=5
            )
            score, _final_state, _hist = run_simulation(
                env, agent, evaluator, init_state, max_steps=max_steps, verbose=False
            )

            rows.append({
                "trial": t,
                "agent": name,
                "score": score,
                "width": meta["width"],
                "height": meta["height"],
                "obstacles": len(meta["obstacles"]),
                "dirt_initial": sum(1 for v in init_state.dirt.values() if v),
            })

    df = pd.DataFrame(rows)
    return df

df214 = run_ex214_trials(trials=60, max_steps=150, seed=7, obstacle_prob=0.18, dirt_prob=0.55)

display(df214.groupby("agent")["score"].agg(["mean", "std", "min", "max"]).sort_values("mean", ascending=False))


## Exercise 2.14 — Results and Discussion

Results below are from 60 trials on randomly generated 4–7 × 4–7 grids
(seed=7, obstacle_prob=0.18, dirt_prob=0.55, max_steps=150).

| Agent | Mean Score | Std Dev | Min | Max |
|---|---|---|---|---|
| Berkeley Reflex | 2462.25 | 712.20 | 1044 | 4280 |
| Berkeley Model-Based | 2322.38 | 817.71 | 894 | 4425 |
| Berkeley Random | 1970.75 | 624.79 | 884 | 3407 |

### Agent descriptions (as implemented)

- **Berkeley Reflex** (`BerkeleyReflexVacuumAgent`): if dirty → SUCK, else pick a
  random movement direction. No memory of visited cells or obstacles.
- **Berkeley Model-Based** (`BerkeleyModelBasedVacuumAgent`): if dirty → SUCK, else
  explore systematically using a learned transition model and DFS backtracking.
  Remembers blocked directions and avoids retrying them.
- **Berkeley Random** (`BerkeleyRandomVacuumAgent`): picks uniformly from all six
  actions (LEFT, RIGHT, UP, DOWN, SUCK, NOOP) regardless of the percept.


In [ ]:
def make_corridor_environment(length=12, dirt_prob=0.6, seed=1):
    rng = random.Random(seed)
    width, height = length, 1
    obstacles = set()  # none inside
    locations, adjacency = build_grid_adjacency(width, height, obstacles)
    env = VacuumEnvironment(locations=locations, adjacency=adjacency, sensor_model=SensorModel())
    agent_loc = locations[0]
    dirt_map = {loc: (rng.random() < dirt_prob) for loc in locations}
    init_state = env.initial_state(agent_loc, dirt_map)
    return env, init_state

env_bad, init_bad = make_corridor_environment(length=18, dirt_prob=0.7, seed=3)

agents = [
    ("Randomized reflex (suck-if-dirty)", BerkeleyReflexVacuumAgent()),
    ("Pure random", BerkeleyRandomVacuumAgent(seed=0)),
    ("Reflex w/ state (explore)", BerkeleyModelBasedVacuumAgent()),
]

rows = []
for name, agent in agents:
    evaluator = PerformanceEvaluator(clean_reward_per_square=1, action_cost=1, clean_event_bonus=5)
    score, _final_state, hist = run_simulation(env_bad, agent, evaluator, init_bad, max_steps=200, verbose=False)
    rows.append((name, score))

pd.DataFrame(rows, columns=["agent", "score"]).sort_values("score", ascending=False)


## Corridor (Adversarial) Environment — Results and Discussion

The corridor test uses an 18×1 grid (a long hallway with no obstacles) and runs each
agent for 200 steps with dirt_prob=0.7. This is a deliberately adversarial environment
for stateless agents.

| Agent | Score |
|---|---|
| Reflex w/ state (explore) | 3279 |
| Randomized reflex (suck-if-dirty) | 1848 |
| Pure random | 1458 |